# 04 – IDS baseline (AI-4)
Cùng một mô hình IDS (Random Forest, cùng tham số) được huấn luyện trên:
1. dữ liệu thật,
2. dữ liệu thật + dữ liệu sinh thô (tất cả các lớp, hoặc chỉ lớp hiếm),
3. dữ liệu thật + dữ liệu sinh đã lọc theo rủi ro (khi nhóm Cyber Security có điểm rủi ro).

Tất cả đánh giá trên **cùng một tập kiểm thử thật** chưa dùng để huấn luyện generator. Chỉ số: precision, recall, F1 (trung bình macro), PR-AUC, và recall từng lớp (nhất là lớp hiếm).

Cần có sẵn dữ liệu sinh trong `outputs/` (notebook 02 và/hoặc 03).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src import config, data

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

DATASET = "unsw_nb15"   # đổi thành "cicids2017" để chạy dataset kia

from src import ids

CAP_PER_CLASS = 3000     # PHẢI bằng giá trị đã dùng khi huấn luyện (500 nếu bạn chạy SMOKE_TEST)
RARE_BELOW = 2000        # lớp có ít hơn số dòng này trong dữ liệu huấn luyện GỐC được coi là "hiếm"
SEEDS = [42]             # đặt [42, 43, 44, 45, 46] để có trung bình ± độ lệch chuẩn (nên làm khi báo cáo)
POSTPROCESS = False      # True = kéo dữ liệu sinh về khoảng giá trị thật trước khi dùng (nên chạy cả hai và so sánh)


## 1. Dữ liệu

In [ ]:
train, test_raw = data.load(DATASET)
real, LABEL, DISCRETE = data.prepare_for_ctgan(train, DATASET, cap_per_class=CAP_PER_CLASS)
test = data.prepare_test(test_raw, DATASET, real.columns)
print("Huấn luyện thật:", real.shape, "| Kiểm thử thật:", test.shape)

raw_counts = train[LABEL].value_counts()   # đếm trên dữ liệu gốc, không phụ thuộc CAP_PER_CLASS
rare_classes = [c for c, n in raw_counts.items() if n < RARE_BELOW]
print("Lớp hiếm dùng để tăng cường:", rare_classes)

synths = {}
for name, fname in {"CTGAN": f"synthetic_ctgan_{DATASET}.csv", "TVAE": f"synthetic_tvae_{DATASET}.csv"}.items():
    f = config.OUT_DIR / fname
    if f.exists():
        synths[name] = pd.read_csv(f)
        if POSTPROCESS:
            from src import evaluation as ev
            synths[name] = ev.clip_to_real_range(real, synths[name], DISCRETE)
        print(f"{name}: {synths[name].shape}" + (" (đã hậu xử lý)" if POSTPROCESS else ""))
    else:
        print(f"{name}: chưa có file {fname}, bỏ qua")

## 2. Các thí nghiệm
Mỗi thí nghiệm là một bảng huấn luyện khác nhau. Dòng "filtered" chỉ chạy khi có file điểm rủi ro: một cột `risk_score`, cùng thứ tự dòng với file dữ liệu sinh,
đặt tên `outputs/risk_scores_<ctgan|tvae>_<dataset>.csv`. Ngưỡng `RISK_THRESHOLD` phải được chọn bằng validation, không chọn tuỳ ý.

In [ ]:
RISK_THRESHOLD = 0.7   # giữ các dòng có risk_score < ngưỡng (ví dụ; chọn bằng validation)

experiments = {"Real only": real}
for name, s in synths.items():
    experiments[f"Real + {name} (all)"] = pd.concat([real, s], ignore_index=True)
    experiments[f"Real + {name} (rare classes only)"] = pd.concat([real, s[s[LABEL].isin(rare_classes)]], ignore_index=True)

    rf = config.OUT_DIR / f"risk_scores_{name.lower()}_{DATASET}.csv"
    if rf.exists():
        risk = pd.read_csv(rf)["risk_score"]
        assert len(risk) == len(s), "Số dòng risk_score phải bằng số dòng dữ liệu sinh"
        kept = s[(risk < RISK_THRESHOLD).to_numpy()]
        experiments[f"Real + {name} (filtered, risk<{RISK_THRESHOLD})"] = pd.concat([real, kept], ignore_index=True)
    else:
        print(f"{name}: chưa có điểm rủi ro ({rf.name}), bỏ qua thí nghiệm filtered (chờ nhóm Cyber Security, AI-5)")

for k, v in experiments.items():
    print(f"{k:50s} {len(v):>8,d} dòng")

## 3. Chạy IDS và so sánh

In [ ]:
summaries, recalls = [], []
for s in SEEDS:
    sm, rc = ids.run_experiments(experiments, test, LABEL, DISCRETE, seed=s)
    summaries.append(sm); recalls.append(rc.drop(columns="test_support").assign(seed=s))
support = rc["test_support"]

summary_mean = pd.concat(summaries).groupby(level=0, sort=False).mean(numeric_only=True)
print(f"Kết quả trung bình trên {len(SEEDS)} seed (chỉ thay đổi tính ngẫu nhiên của Random Forest, không phải của generator)")
display(summary_mean.round(4))
if len(SEEDS) > 1:
    print("Độ lệch chuẩn giữa các seed:")
    display(pd.concat(summaries).groupby(level=0, sort=False).std(numeric_only=True).round(4))

## 4. Recall từng lớp (nhất là lớp hiếm)

In [ ]:
recall_by_class = pd.concat(recalls).groupby(level=0).mean(numeric_only=True).drop(columns="seed")
recall_by_class.insert(0, "test_support", support)
recall_by_class = recall_by_class.sort_values("test_support")
display(recall_by_class.round(3))

rare_view = recall_by_class.loc[[c for c in recall_by_class.index if c in rare_classes]].drop(columns="test_support")
if len(rare_view):
    rare_view.T.plot(kind="bar", figsize=(10, 4), title="Recall của các lớp hiếm theo thí nghiệm")
    plt.ylabel("recall"); plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

In [ ]:
config.OUT_DIR.mkdir(exist_ok=True)
summary_mean.round(4).to_csv(config.OUT_DIR / f"ids_baseline_summary_{DATASET}.csv")
recall_by_class.round(4).to_csv(config.OUT_DIR / f"ids_baseline_recall_by_class_{DATASET}.csv")
print("Đã lưu kết quả vào outputs/")

## Cách đọc kết quả
- So "Real + ... (all)" và "(rare classes only)" với "Real only": recall lớp hiếm có tăng không, và macro F1 có giảm không?
- Chênh lệch nhỏ có thể chỉ là ngẫu nhiên. Chạy nhiều seed trước khi kết luận, và lưu ý các lớp hiếm trong tập kiểm thử rất ít bản ghi nên recall dao động mạnh.
- Dữ liệu sinh nghèo ở lớp hiếm (xem notebook 04) thì không thể cải thiện recall lớp đó.
- Đây là so sánh sơ bộ trên UNSW-NB15 và chưa dùng quy tắc chia dữ liệu chính thức (DS2).